# 0l · **LIBERO-10 결과 수집** — `eval_clean/libero_10`

`outputs/final/eval_clean/libero_10/` 아래에 쌓인 eval 결과를 **있는 그대로 스캔**해서
SR 표 · run 목록 · action(.pt) 을 모아 CSV/zip 으로 뽑는다.

레이아웃을 가정하지 않는다 — `<model>/seed<N>/` 든 `<model>/seed<N>/rep<r>/` 든
`eval_info.json` 이 있는 폴더를 전부 찾아 읽는다.


In [ ]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)
import numpy as np
from collections import defaultdict

TASK = 'libero_10'
ROOT = cf.OUTPUT_BASE / 'eval_clean' / TASK
OUT  = cf.OUTPUT_BASE / 'share' / TASK
OUT.mkdir(parents=True, exist_ok=True)

print('scan:', ROOT)
print('exists:', ROOT.is_dir())
if not ROOT.is_dir():
    print('!! that path is missing - check LEROBOT_OUTPUT / paths')

## 1) 폴더 스캔 — `eval_info.json` 이 있는 모든 run


In [ ]:
def parse_info(p):
    try:
        d = json.loads(p.read_text())
    except Exception:
        return None
    ov = d.get('overall', d)                       # schema may nest under 'overall'
    sr = ov.get('pc_success')
    if sr is None and 'success' in ov:
        sr = 100.0 * float(np.mean(ov['success']))
    return {'sr': sr, 'n_episodes': ov.get('n_episodes')}

runs = []
for info in sorted(ROOT.rglob('eval_info.json')):
    parts = info.parent.relative_to(ROOT).parts   # e.g. ('acm','seed0') or ('acm','seed0','rep2')
    model = parts[0] if parts else '?'
    seed = next((p for p in parts if p.startswith('seed')), '?')
    rep = next((p for p in parts[1:] if 'rep' in p and p != seed), '')
    m = parse_info(info)
    if m is None:
        print('  parse failed:', info.parent.relative_to(ROOT)); continue
    runs.append({'model': model, 'seed': seed, 'rep': rep,
                 'path': str(info.parent.relative_to(ROOT)),
                 'sr': m['sr'], 'n_episodes': m['n_episodes'],
                 'has_actions': (info.parent / 'actions').is_dir()})

print('runs found:', len(runs))
print()
print(f"{'model':<12}{'seed':>7}{'rep':>8}{'SR':>8}{'n_ep':>7}{'act':>5}")
print('-' * 50)
for r in runs:
    sr = f"{r['sr']:.1f}" if r['sr'] is not None else '-'
    nep = r['n_episodes'] if r['n_episodes'] is not None else '-'
    print(f"{r['model']:<12}{r['seed']:>7}{(r['rep'] or '-'):>8}{sr:>8}{nep:>7}{('Y' if r['has_actions'] else '-'):>5}")

## 2) 원본 CSV — 모든 run 그대로


In [ ]:
if runs:
    keys = ['model', 'seed', 'rep', 'sr', 'n_episodes', 'has_actions', 'path']
    with open(OUT / 'libero_runs.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
        for r in runs:
            w.writerow({k: r[k] for k in keys})
    print('saved:', OUT / 'libero_runs.csv')
else:
    print('no runs - check the scan above')

## 3) model × seed 표 (rep 이 있으면 열로 펼침 + avg)


In [ ]:
cellmap = defaultdict(dict)          # (model, seed) -> {rep: sr}
reps_seen = set()
for r in runs:
    if r['sr'] is None:
        continue
    rp = r['rep'] or 'run'
    cellmap[(r['model'], r['seed'])][rp] = r['sr']
    reps_seen.add(rp)
reps_sorted = sorted(reps_seen)

def fmt(v, d=1):
    return '-' if v is None else f'{v:.{d}f}'

table = []
for (model, seed) in sorted(cellmap):
    d = cellmap[(model, seed)]
    got = [d[rp] for rp in reps_sorted if rp in d]
    row = {'model': model, 'seed': seed}
    for rp in reps_sorted:
        row[rp] = round(d[rp], 1) if rp in d else None
    row['avg'] = round(float(np.mean(got)), 2) if got else None
    table.append(row)

cols = ['model', 'seed'] + reps_sorted + ['avg']
with open(OUT / 'libero_sr_table.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(table)

hdr = f"{'model':<12}{'seed':>7}" + ''.join(f'{rp:>9}' for rp in reps_sorted) + f"{'avg':>9}"
print(hdr)
print('-' * len(hdr))
for row in table:
    line = f"{row['model']:<12}{row['seed']:>7}"
    for rp in reps_sorted:
        line += f'{fmt(row[rp]):>9}'
    line += f'{fmt(row["avg"], 2):>9}'
    print(line)
print()
print('saved:', OUT / 'libero_sr_table.csv')

## 4) 모델별 요약 (전 seed/rep pooled)


In [ ]:
by_model = defaultdict(list)
for r in runs:
    if r['sr'] is not None:
        by_model[r['model']].append(r['sr'])

print(f"{'model':<14}{'mean SR':>9}{'std':>7}{'n_run':>7}")
print('-' * 37)
sum_rows = []
for m in sorted(by_model):
    v = by_model[m]
    mean, std = float(np.mean(v)), float(np.std(v))
    sum_rows.append({'model': m, 'mean_SR': round(mean, 2), 'std': round(std, 2), 'n_run': len(v)})
    print(f"{m:<14}{mean:>9.2f}{std:>7.2f}{len(v):>7}")

if sum_rows:
    with open(OUT / 'libero_summary.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'mean_SR', 'std', 'n_run'])
        w.writeheader(); w.writerows(sum_rows)
    print()
    print('saved:', OUT / 'libero_summary.csv')

## 5) 떨림 지표 — **jerk RMS · LDJ · SPARC · Sign Flip** (action.pt 있을 때)
`actions/` 가 있는 run 의 궤적을 pool 해 계산. libero fps=30.


In [ ]:
trajs = {}
for r in runs:
    if not r['has_actions']:
        continue
    tr = cf.v23._load_action_trajs(ROOT / r['path'] / 'actions') or []
    if tr:
        trajs.setdefault(r['model'], []).extend(tr)

if not trajs:
    print('no action(.pt) - SR/tables only (check the eval wrote RECORD_DIR)')
else:
    import smooth_metrics as sm
    # 은지님 요청 지표: jerk RMS, LDJ, SPARC, Sign Flip
    print(f"{'model':<14}{'jerk_RMS':>10}{'LDJ':>10}{'SPARC':>10}{'sign_flip':>11}{'n_ep':>6}")
    print('  (smoother =    lower      higher    near-0       lower)')
    print('-' * 61)
    jrows = []
    for m in sorted(trajs):
        agg = sm.aggregate_smoothness(trajs[m], chunk_size=100, fs=cf.fps_of(TASK))
        row = {'model': m,
               'jerk_rms': round(agg['jerk_rms_mean'], 5),
               'jerk_rms_std': round(agg['jerk_rms_std'], 5),
               'LDJ': round(agg['ldj_mean'], 3),
               'LDJ_std': round(agg['ldj_std'], 3),
               'SPARC': round(agg['sparc_mean'], 3),
               'SPARC_std': round(agg['sparc_std'], 3),
               'sign_flips': round(agg['sign_flips_mean'], 4),
               'sign_flips_std': round(agg['sign_flips_std'], 4),
               'n_episodes': agg['n_episodes']}
        jrows.append(row)
        print(f"{m:<14}{row['jerk_rms']:>10.4f}{row['LDJ']:>10.2f}{row['SPARC']:>10.2f}"
              f"{row['sign_flips']:>11.3f}{row['n_episodes']:>6}")
    with open(OUT / 'libero_smoothness.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(jrows[0])); w.writeheader(); w.writerows(jrows)
    print()
    print('jerk RMS : per-step jerk 크기 (작을수록 매끄러움)')
    print('LDJ      : log dimensionless jerk (클수록 = 0 에 가까울수록 매끄러움)')
    print('SPARC    : spectral arc length (0 에 가까울수록 매끄러움)')
    print('sign_flip: 방향 반전 횟수 / step (작을수록 덜 떨림)')
    print('saved:', OUT / 'libero_smoothness.csv')

## 6) 전부 zip 으로 → 팀에 이 파일 하나


In [ ]:
import shutil
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'{TASK}_collect'), 'zip', root_dir=OUT)
print('send this file:', zip_path)
for p in sorted(Path(OUT).rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')